<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [4]</a>'.</span>

# 03 — Model Training

Цель ноутбука: показать конфиг обучения, размеры train/val/test, архитектуру модели, историю loss и лучший checkpoint.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_SYNTHETIC = ROOT / "data" / "synthetic"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

print("Project root:", ROOT)
print("Synthetic data:", DATA_SYNTHETIC)

Project root: /Users/andrey/Documents/projects/COMPASS-AI
Synthetic data: /Users/andrey/Documents/projects/COMPASS-AI/data/synthetic


In [2]:
split_meta_path = DATA_PROCESSED / "split_metadata.json"
training_config_path = REPORTS / "training_config.json"
history_path = REPORTS / "training_history.csv"
checkpoint_path = ROOT / "models" / "compass_matching_model.pt"

with open(split_meta_path, "r", encoding="utf-8") as file:
    split_meta = json.load(file)

with open(training_config_path, "r", encoding="utf-8") as file:
    training_config = json.load(file)

print("split metadata:")
display(split_meta)

print("training config:")
display(training_config)

print("checkpoint exists:", checkpoint_path.exists(), checkpoint_path)

split metadata:


{'random_state': 42,
 'split_strategy': 'group_split_by_task_id',
 'train_size_target': 0.7,
 'val_size_target': 0.15,
 'test_size_target': 0.15,
 'splits': [{'split': 'train',
   'rows': 42052,
   'tasks': 1750,
   'employees': 36,
   'success_rate': 0.450775},
  {'split': 'val',
   'rows': 9024,
   'tasks': 375,
   'employees': 36,
   'success_rate': 0.44293},
  {'split': 'test',
   'rows': 8924,
   'tasks': 375,
   'employees': 36,
   'success_rate': 0.457642}],
 'leakage_check': {'task_id_overlap_train_val': 0,
  'task_id_overlap_train_test': 0,
  'task_id_overlap_val_test': 0}}

training config:


{'device': 'mps',
 'epochs': 40,
 'batch_size': 512,
 'learning_rate': 0.0007,
 'weight_decay': 0.0001,
 'hidden_dim': 256,
 'embedding_dim': 128,
 'dropout': 0.1,
 'patience': 8,
 'train_rows': 42052,
 'val_rows': 9024,
 'task_dim': 404,
 'employee_dim': 60,
 'pair_dim': 13}

checkpoint exists: True /Users/andrey/Documents/projects/COMPASS-AI/models/compass_matching_model.pt


In [3]:
train_df = pd.read_parquet(DATA_PROCESSED / "train.parquet")
val_df = pd.read_parquet(DATA_PROCESSED / "val.parquet")
test_df = pd.read_parquet(DATA_PROCESSED / "test.parquet")

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "tasks": train_df["task_id"].nunique(),
            "success_rate": train_df["success_label"].mean(),
        },
        {
            "split": "val",
            "rows": len(val_df),
            "tasks": val_df["task_id"].nunique(),
            "success_rate": val_df["success_label"].mean(),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "tasks": test_df["task_id"].nunique(),
            "success_rate": test_df["success_label"].mean(),
        },
    ]
)

display(split_summary)

,split,rows,tasks,success_rate
0,train,42052,1750,0.450775
1,val,9024,375,0.442930
2,test,8924,375,0.457642


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [4]:
from src.models.matching_net import TaskEmployeeMatchingNet

model = TaskEmployeeMatchingNet(
    task_input_dim=training_config["task_dim"],
    employee_input_dim=training_config["employee_dim"],
    pair_input_dim=training_config["pair_dim"],
    hidden_dim=training_config.get("hidden_dim", 256),
    embedding_dim=training_config.get("embedding_dim", 128),
    dropout=training_config.get("dropout", 0.1),
)

model

TypeError: TaskEmployeeMatchingNet.__init__() got an unexpected keyword argument 'task_input_dim'

In [ ]:
history = pd.read_csv(history_path)
display(history.head())

In [ ]:
fig = px.line(
    history,
    x="epoch",
    y=["train_loss", "val_loss"],
    title="Training and validation loss",
)
fig.show()
fig.write_html(FIGURES / "training_loss.html")

In [ ]:
if "val_roc_auc" in history.columns:
    fig = px.line(history, x="epoch", y="val_roc_auc", title="Validation ROC-AUC")
    fig.show()
    fig.write_html(FIGURES / "training_val_roc_auc.html")